## The Price Is Right - Building a Deal Scanner with Structured Outputs and Pydantic

In [1]:
# libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests

In [2]:
# setting the environment
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:29<00:00,  9.90s/it]


In [4]:
len(deals)

30

In [5]:
deals[2].describe()

'Title: Lisen 10-Foot 240W USB-C to USB-C Fast Charging Cable for $10 + free shipping w/ Prime\nDetails: This Lisen 10-foot USB-C cable is 50% off at Amazon. It supports up to 240W charging and works with phones, tablets, and laptops that use USB-C. The silicone cable includes a cable tie for storage. Its price is below its 90-day Amazon average of $12. Shipping is free for Prime members. Buy Now at Amazon\nFeatures: 10-foot length Supports up to 240W USB-C Power Delivery charging Silicone cable body rated for 50,000+ bends Dual E-Marker chips for over-current, voltage, & temperature protection Includes a hook-and-loop cable tie 12-month warranty\nURL: https://www.dealnews.com/Lisen-10-Foot-240-W-USB-C-to-USB-C-Fast-Charging-Cable-for-10-free-shipping-w-Prime/22325137.html?iref=rss-c142'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more"

In [7]:
# function to make suitable user prompt given scraped deals
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join(scrape.describe() for scrape in scraped)
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Open-Box Unlocked Motorola Moto G Stylus 5G 256GB Smartphone (2024) for $133 + free shipping
Details: Walmart offers the open-box Unlocked Motorola Moto G Stylus 5G 256GB Smartphone for $132.61.  That's $67 under the cost of a factory-sealed unit.  Shipping is free. Buy Now at Walmart
Features: Built-in stylus 6.7" pOLED display 256GB of storage 50MP main camera Android 14
URL:

In [9]:
response = openai.chat.completions.parse(
    model=MODEL,
    messages=messages,
    response_format=DealSelection,
    reasoning_effort='minimal'
)

results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Open-box unlocked Motorola Moto G Stylus 5G (2024) with a built-in stylus designed for note-taking and sketching. It features a 6.7-inch pOLED display for vibrant colors, 256GB of onboard storage for apps and media, a 50MP main camera for high-resolution photos, and runs Android 14. The unit is sold unlocked for use on compatible carriers and includes the phone hardware in open-box condition.', price=132.61, url='https://www.dealnews.com/products/Motorola/Unlocked-Motorola-Moto-G-Stylus-5-G-256-GB-Smartphone-2024/551437.html?iref=rss-c142'), Deal(product_description='Hisense M7 65M7SG 65-inch 4K QLED smart TV with miniLED backlighting for improved contrast and brightness. The set delivers 4K resolution with Dolby Vision IQ, HDR10+, HDR10, and HLG support, includes a built-in subwoofer with Dolby Atmos support for enhanced audio, runs Google TV for streaming apps, and offers four HDMI inputs for multiple devices.', price=374.0, url='https:/

In [10]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

Open-box unlocked Motorola Moto G Stylus 5G (2024) with a built-in stylus designed for note-taking and sketching. It features a 6.7-inch pOLED display for vibrant colors, 256GB of onboard storage for apps and media, a 50MP main camera for high-resolution photos, and runs Android 14. The unit is sold unlocked for use on compatible carriers and includes the phone hardware in open-box condition.
132.61
https://www.dealnews.com/products/Motorola/Unlocked-Motorola-Moto-G-Stylus-5-G-256-GB-Smartphone-2024/551437.html?iref=rss-c142

Hisense M7 65M7SG 65-inch 4K QLED smart TV with miniLED backlighting for improved contrast and brightness. The set delivers 4K resolution with Dolby Vision IQ, HDR10+, HDR10, and HLG support, includes a built-in subwoofer with Dolby Atmos support for enhanced audio, runs Google TV for streaming apps, and offers four HDMI inputs for multiple devices.
374.0
https://www.dealnews.com/products/Hisense/Hisense-M7-65-M7-SG-65-4-K-QLED-Smart-Google-TV/551378.html?iref=rss

In [11]:
root = logging.getLogger()
root.setLevel(logging.INFO)

### Creating the Scanning Agent

In [12]:
from agents.scanner_agent import ScannerAgent

In [13]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [14]:
result

DealSelection(deals=[Deal(product_description='Open-box 2024 Motorola Moto G Stylus 5G unlocked smartphone with a built-in stylus and a large 6.7-inch pOLED display. It includes 256 GB of onboard storage, runs Android 14, and has a 50 MP main camera for detailed photos. The phone is a full-featured midrange device aimed at productivity with stylus input and generous storage.', price=132.61, url='https://www.dealnews.com/products/Motorola/Unlocked-Motorola-Moto-G-Stylus-5-G-256-GB-Smartphone-2024/551437.html?iref=rss-c142'), Deal(product_description='Hisense M7 65M7SG is a 65-inch 4K QLED smart TV featuring mini-LED backlighting for improved contrast and brightness. It supports Dolby Vision IQ, HDR10+, HDR10, and HLG, and includes a built-in subwoofer with Dolby Atmos support for enhanced audio. The set runs Google TV and provides four HDMI inputs for multiple source connections.', price=374.0, url='https://www.dealnews.com/products/Hisense/Hisense-M7-65-M7-SG-65-4-K-QLED-Smart-Google-T

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone. We need to set two things in the `.env`

In [15]:
# loading the keys
load_dotenv(override=True)

True

In [16]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = 'https://api.pushover.net/1/messages.json'

In [17]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")


if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [18]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
load_dotenv(override=True)

In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push('SUCH A MASSIVE DEAL!!')

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [20]:
agent.notify("A special deal on Samsung 60 inch LED TV going at a great bargain", 300, 1000, 'www.samsung.com')

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message


03:15:20 - LiteLLM:INFO: utils.py:4502 - 
LiteLLM completion() model= gpt-5-mini; provider = openai


INFO:LiteLLM:
LiteLLM completion() model= gpt-5-mini; provider = openai


03:15:30 - LiteLLM:INFO: utils.py:1864 - Wrapper: Completed Call, calling success_handler


INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
